# Run the Titanic Serverless Spark Inference Pipeline

Submit [titanic-spark-inference-pipeline.yaml](./titanic-spark-inference-pipeline.yaml) directly through the Azure ML Python SDK. Its single managed Spark 3.5 job loads the immutable artifact of the registered Titanic model and writes predictions to a unique ADLS directory.

No endpoint, deployment, custom environment, AML compute cluster, or local Spark runtime is used. Existing endpoints are left unchanged.

In [ ]:
from pathlib import Path
import json
import os
import sys
import uuid

import pandas as pd
from azure.ai.ml import Input, MLClient, Output, load_job
from azure.ai.ml.constants import AssetTypes
from azure.identity import AzureCliCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv


def find_workshop_root() -> Path:
    notebook_file = globals().get("__vsc_ipynb_file__")
    start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENDA.md").is_file() and (candidate / ".env.example").is_file():
            return candidate
        nested = candidate / "workshop"
        if (nested / "AGENDA.md").is_file() and (nested / ".env.example").is_file():
            return nested
    raise FileNotFoundError("Run this notebook from inside the workshop folder")


def require_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in workshop/.env")
    return value


def env_flag(name: str) -> bool:
    value = os.getenv(name, "false").strip().lower()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be exactly true or false")
    return value == "true"


WORKSHOP_ROOT = find_workshop_root()
MODULE_ROOT = WORKSHOP_ROOT / "notebooks" / "06_spark_jobs_pipelines"
SOURCE_ROOT = MODULE_ROOT / "src"
PIPELINE_PATH = MODULE_ROOT / "titanic-spark-inference-pipeline.yaml"
OUTPUT_ROOT = WORKSHOP_ROOT / "outputs" / "titanic_spark_inference"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from inference_verification import (
    assert_native_spark_job,
    assert_prediction_parity,
    download_predictions,
    load_verified_predictions,
)
from registered_model_artifacts import stage_registered_model
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

SUBSCRIPTION_ID = require_env("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = require_env("AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = require_env("AZUREML_WORKSPACE_NAME")
MODEL_NAME = require_env("TITANIC_SPARK_MODEL_NAME")
MODEL_VERSION = require_env("TITANIC_SPARK_MODEL_VERSION")
STORAGE_ACCOUNT = require_env("TITANIC_SPARK_STORAGE_ACCOUNT")
FILESYSTEM = require_env("TITANIC_SPARK_FILESYSTEM")
OUTPUT_PREFIX = require_env("TITANIC_SPARK_INFERENCE_OUTPUT_PATH").strip("/")
EXISTING_JOB_NAME = os.getenv("TITANIC_SPARK_INFERENCE_JOB_NAME", "").strip()
REFERENCE_PATH = os.getenv("TITANIC_SPARK_INFERENCE_REFERENCE_PREDICTIONS_PATH", "").strip()
RUN_PIPELINE = env_flag("RUN_TITANIC_SPARK_INFERENCE_PIPELINE")

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None, process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
registered_model = ml_client.models.get(MODEL_NAME, version=MODEL_VERSION)
if registered_model.type != AssetTypes.CUSTOM_MODEL or not registered_model.path:
    raise TypeError("Expected a registered custom model with an immutable artifact URI")
model_uri = registered_model.path
input_uri = f"abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/data/titanic.csv"

print(f"Workspace: {WORKSPACE_NAME}")
print(f"Pipeline YAML: {PIPELINE_PATH}")
print(f"Registered model: {registered_model.name}:{registered_model.version}")
print(f"Immutable model URI: {model_uri}")
print(f"Run enabled: {RUN_PIPELINE}; existing job: {EXISTING_JOB_NAME or 'none'}")

In [ ]:
pipeline_job = load_job(PIPELINE_PATH)
if list(pipeline_job.jobs) != ["spark_score"]:
    raise ValueError("Inference must contain exactly one native Spark scoring job")
spark_node = pipeline_job.jobs["spark_score"]
if spark_node.type != "spark" or spark_node.compute is not None:
    raise ValueError("Inference must use serverless Spark, not an AML compute cluster")
if pipeline_job.settings.default_compute is not None:
    raise ValueError("The inference pipeline must not assign a default AML compute cluster")
if str(spark_node.resources.runtime_version) != "3.5":
    raise ValueError("Inference requires native Spark runtime 3.5")

pipeline_job.name = f"titanic-spark-inference-{uuid.uuid4().hex[:12]}"
output_uri = (
    f"abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"{OUTPUT_PREFIX}/{pipeline_job.name}/"
)
pipeline_job.inputs.titanic_data = Input(type=AssetTypes.URI_FILE, path=input_uri, mode="direct")
pipeline_job.inputs.model_input = Input(type=AssetTypes.URI_FOLDER, path=model_uri, mode="direct")
pipeline_job.inputs.model_asset_id = registered_model.id
pipeline_job.outputs.predictions = Output(type=AssetTypes.URI_FOLDER, path=output_uri, mode="direct")
pipeline_job.settings.force_rerun = True
pipeline_job.tags["registered_model_id"] = registered_model.id

print(f"Native Spark: {spark_node.resources.runtime_version}; instance={spark_node.resources.instance_type}")
print(f"Input: {input_uri}")
print(f"Unique predictions output: {output_uri}")

In [ ]:
final_job = None
if RUN_PIPELINE:
    if EXISTING_JOB_NAME:
        submitted_job = ml_client.jobs.get(EXISTING_JOB_NAME)
        print(f"Reusing inference pipeline: {submitted_job.name}")
    else:
        data_lake = DataLakeServiceClient(
            account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net", credential=credential,
        )
        model_uri = stage_registered_model(
            ml_client, registered_model, credential,
            data_lake.get_file_system_client(FILESYSTEM), STORAGE_ACCOUNT, FILESYSTEM,
        )
        pipeline_job.inputs.model_input = Input(type=AssetTypes.URI_FOLDER, path=model_uri, mode="direct")
        print(f"Checksum-verified registered model copy: {model_uri}")
        submitted_job = ml_client.jobs.create_or_update(pipeline_job)
        print(f"Serverless inference pipeline: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Serverless Spark inference ended with status {final_job.status}")
    print(f"Studio: {final_job.studio_url}")
else:
    print("Submission disabled. Set RUN_TITANIC_SPARK_INFERENCE_PIPELINE=true to run inference.")

In [ ]:
if RUN_PIPELINE:
    if final_job is None or final_job.status != "Completed":
        raise RuntimeError("A completed inference pipeline is required before verification")
    children = list(ml_client.jobs.list(parent_job_name=final_job.name))
    if len(children) != 1:
        raise RuntimeError(f"Expected one native Spark child, found {len(children)}")
    spark_job = ml_client.jobs.get(children[0].name)
    if spark_job.status != "Completed":
        raise RuntimeError(f"Spark child did not complete: {spark_job.status}")
    assert_native_spark_job(spark_job)
    print(
        f"Verified native Spark child: {spark_job.name}; compute={spark_job.compute}; "
        f"ComputeTargetType={spark_job.properties['_azureml.ComputeTargetType']}"
    )
    prediction_output = (final_job.outputs or {}).get("predictions")
    if prediction_output is None or not prediction_output.path:
        raise RuntimeError("The completed pipeline did not expose its named predictions output")
    download_dir = OUTPUT_ROOT / final_job.name
    download_predictions(prediction_output.path, download_dir, credential)
    source = pd.read_csv(MODULE_ROOT / "data" / "titanic.csv")
    expected_ids = set(source["PassengerId"].astype(int))
    predictions, manifest = load_verified_predictions(
        download_dir, expected_ids, registered_model.id, registered_model.path,
    )
    if REFERENCE_PATH:
        reference = pd.read_csv(
            WORKSHOP_ROOT / REFERENCE_PATH, sep=r"\s+", header=None,
            names=["source_file", "PassengerId", "prediction", "actual_label", "survival_probability"],
        )
        assert_prediction_parity(predictions, reference)
        print("Prediction parity passed against registered-model reference output")
    else:
        print("Parity comparison disabled; configure TITANIC_SPARK_INFERENCE_REFERENCE_PREDICTIONS_PATH to enable it")
    predictions_path = download_dir / "predictions.csv"
    predictions.sort_values("PassengerId").to_csv(predictions_path, index=False)
    print(json.dumps(manifest, indent=2, sort_keys=True))
    print(predictions.head(10).to_string(index=False))
    print(f"Validated serverless Spark predictions: {len(predictions)} rows")
    print(f"Predictions: {predictions_path}")
    print(f"Final serverless inference status: {final_job.status}")
else:
    print("No inference requested; prediction verification was not run.")

## Expected result and safe reruns

The pipeline starts one `amlspark` child on serverless E4s v3 resources. Its manifest proves Spark 3.5, a non-local master, and the selected registered model. The Titanic sample produces 891 unique class/probability rows.

Set `RUN_TITANIC_SPARK_INFERENCE_PIPELINE=false` after execution. Set `TITANIC_SPARK_INFERENCE_JOB_NAME` to the successful run to download and re-check results without submitting another job. Keep the actual saved notebook outputs.